# Chatbot with Sentence Classifier

This notebook demonstrates a chatbot architecture with:
- Intent classification using machine learning
- Conversation context management
- Slot filling for structured information
- Fallback mechanisms for low-confidence inputs
- Conversation logic and follow-ups

## 1. Setup and Initialization

In [ ]:
import os
import sys

# Add chatbot module to path
sys.path.insert(0, os.getcwd())

from chatbot import Chatbot
from conversation_context import ConversationContext
import json

print('Importing chatbot components...')
# Create models directory if it doesn't exist
os.makedirs('models', exist_ok=True)
print('Models directory ready!')

## 2. Initialize Chatbot

In [ ]:
# Initialize chatbot (will load or train model)
chatbot = Chatbot(model_path='models/intent_model.pkl')
print('Chatbot initialized successfully!')
print(f'Intent classifier ready with intents: {chatbot.classifier.intents}')

## 3. Test Intent Classification

In [ ]:
# Test some example messages
test_messages = [
    'hello there',
    'i need help with my order',
    'can i buy a laptop',
    'this is broken',
    'what are your store hours',
    'xyz123abc'  # Low confidence example
]

print('Testing Intent Classification:\n')
for msg in test_messages:
    result = chatbot.classifier.predict(msg)
    print(f'Message: \"{msg}\"')
    print(f'  Intent: {result["intent"]}')
    print(f'  Confidence: {result["confidence"]: .4f}')
    print(f'  Below Threshold: {result["below_threshold"]}\n')

## 4. Simulate Conversation - Greeting Flow

In [ ]:
# Start a conversation
user_id = 'user_001'

conversation_flow_1 = [
    'Hello',
    'I want to place an order'
]

print('=== Conversation Flow 1: Greeting & Order ===' )
print()

for user_msg in conversation_flow_1:
    response, context = chatbot.process_message(user_id, user_msg)
    print(f'User: {user_msg}')
    print(f'Bot: {response}')
    print(f'Intent: {context.current_intent}')
    print(f'Pending Slots: {chatbot.get_pending_slots(user_id)}')
    print('-' * 50)
    print()

## 5. Slot Filling

In [ ]:
# Continue conversation with slot filling
slot_filling_flow = [
    'My name is John',
    'I would like a laptop',
    'I need 2 please',
    'my email is john@example.com'
]

print('=== Conversation Flow 2: Slot Filling ===' )
print()

for user_msg in slot_filling_flow:
    response, context = chatbot.process_message(user_id, user_msg)
    print(f'User: {user_msg}')
    print(f'Bot: {response}')
    print(f'Slots Filled: {[(s, context.slots[s].value) for s in context.slots if context.slots[s].value]}')
    print(f'Pending Slots: {chatbot.get_pending_slots(user_id)}')
    print('-' * 50)
    print()

## 6. Conversation Summary

In [ ]:
# Get conversation summary
summary = chatbot.get_conversation_summary(user_id)
print('=== Conversation Summary ===' )
print(json.dumps(summary, indent=2))

## 7. Fallback Handling - Low Confidence

In [ ]:
# Test fallback mechanism
user_id_fallback = 'user_002'

fallback_flow = [
    'xyz123blah',  # Low confidence
    'qwerty',      # Low confidence
    'asdfgh',      # Low confidence - should escalate
    'can you help' # Recovery
]

print('=== Fallback Handling Flow ===' )
print()

for user_msg in fallback_flow:
    response, context = chatbot.process_message(user_id_fallback, user_msg)
    print(f'User: {user_msg}')
    print(f'Bot: {response}')
    print(f'Escalation Needed: {context.get_context_flag("needs_escalation")}')
    print('-' * 50)
    print()

## 8. Support Flow

In [ ]:
# Test support flow
user_id_support = 'user_003'

support_flow = [
    'hi',
    'my order is late',
    'it was supposed to arrive yesterday'
]

print('=== Support Flow ===' )
print()

for user_msg in support_flow:
    response, context = chatbot.process_message(user_id_support, user_msg)
    print(f'User: {user_msg}')
    print(f'Bot: {response}')
    print(f'In Support Flow: {context.get_context_flag("in_support_flow")}')
    print('-' * 50)
    print()

## 9. Model Evaluation

In [ ]:
print('=== Model Evaluation ===' )
print()
chatbot.evaluate_model()

## 10. Architecture Overview

### Components:

1. **Intent Classifier** (`intent_classifier.py`)
   - Classifies user input into predefined intents
   - Uses TF-IDF + Random Forest
   - Provides confidence scores

2. **Conversation Context** (`conversation_context.py`)
   - Manages conversation state
   - Handles slot management (pending, filled, confirmed)
   - Tracks conversation history
   - Maintains context flags

3. **Response Generator** (`response_generator.py`)
   - Generates contextual responses
   - Handles slot requests
   - Implements fallback mechanisms
   - Supports follow-ups and confirmations

4. **Main Chatbot** (`chatbot.py`)
   - Orchestrates all components
   - Manages conversations per user
   - Implements conversation logic
   - Handles entity extraction

### Key Features:

- **Slot Filling**: Collects required information across turns
- **Fallback Handling**: Gracefully handles low-confidence inputs
- **Escalation**: Transfers to human agent after max failed attempts
- **Conversation Logic**: Different flows for different intents
- **Context Awareness**: Maintains state across multiple turns
- **Confirmations**: Confirms user information before action